# Course 6: Winning Online AI Competitions
## Session 10: Ensembling

**Goal:** blend out-of-fold predictions five ways, check each blend honestly, and choose two final submissions.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. Five different models, same folds
Diversity matters as much as strength: a blend gains when models make different mistakes.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss
X, y = make_classification(n_samples=6000, n_features=30, n_informative=10, n_redundant=5, flip_y=0.08, class_sep=0.8, random_state=0)
Xtr, ytr, Xte, yte = X[:3000], y[:3000], X[3000:], y[3000:]
folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(Xtr, ytr))
models = {"logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "knn": lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(25)),
          "rf": lambda: RandomForestClassifier(200, min_samples_leaf=3, n_jobs=-1, random_state=0),
          "extra_trees": lambda: ExtraTreesClassifier(200, min_samples_leaf=3, n_jobs=-1, random_state=0),
          "hgb": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0)}
oof, test = pd.DataFrame(index=range(len(ytr))), pd.DataFrame(index=range(len(yte)))
for name, make in models.items():
    o, t = np.zeros(len(ytr)), np.zeros(len(yte))
    for tr, va in folds:
        m = make().fit(Xtr[tr], ytr[tr]); o[va] = m.predict_proba(Xtr[va])[:, 1]; t += m.predict_proba(Xte)[:, 1] / 5
    oof[name], test[name] = o, t
    print(f"{name:12s} OOF AUC {roc_auc_score(ytr, o):.4f} | test AUC {roc_auc_score(yte, t):.4f}")
print("correlation between the models' OOF predictions:"); print(oof.corr().round(2))

### 2. Five ways to blend

In [ ]:
from scipy.optimize import minimize
from scipy.stats import rankdata
auc = roc_auc_score
blends = {}

# 1. Simple average
blends["1. average"] = (oof.mean(1).values, test.mean(1).values)

# 2. Rank average: only the order matters, so models with different scales mix cleanly (good for AUC)
rank = lambda df: df.apply(rankdata).values / len(df)
blends["2. rank average"] = (rank(oof).mean(1), rank(test).mean(1))

# 3. Weights found by optimization on the OOF predictions
def neg_auc(w): w = np.abs(w) / np.abs(w).sum(); return -auc(ytr, oof.values @ w)
w_opt = minimize(neg_auc, np.ones(5) / 5, method="Nelder-Mead", options={"maxiter": 400, "xatol": 1e-3}).x; w_opt = np.abs(w_opt) / np.abs(w_opt).sum()
blends["3. optimized weights"] = (oof.values @ w_opt, test.values @ w_opt)

# 4. Hill climbing: start from the best model, keep adding (with repeats) whichever model improves the blend most
def hill_climb(P, y, max_steps=30):
    chosen = [int(np.argmax([auc(y, P[:, j]) for j in range(P.shape[1])]))]; best = auc(y, P[:, chosen].mean(1))
    for _ in range(max_steps):
        scores = [auc(y, P[:, chosen + [j]].mean(1)) for j in range(P.shape[1])]
        if max(scores) <= best + 1e-6: break
        chosen.append(int(np.argmax(scores))); best = max(scores)
    return np.bincount(chosen, minlength=P.shape[1]) / len(chosen)
w_hc = hill_climb(oof.values, ytr); blends["4. hill climbing"] = (oof.values @ w_hc, test.values @ w_hc)

# 5. Stacking: a second-level model trained on the OOF predictions (its own OOF needs a second round of folds)
stack_oof, stack_test = np.zeros(len(ytr)), np.zeros(len(yte))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=1).split(oof, ytr):
    meta = LogisticRegression(C=1.0, max_iter=1000).fit(oof.values[tr], ytr[tr]); stack_oof[va] = meta.predict_proba(oof.values[va])[:, 1]
stack_test = LogisticRegression(C=1.0, max_iter=1000).fit(oof.values, ytr).predict_proba(test.values)[:, 1]
blends["5. stacking (logistic)"] = (stack_oof, stack_test)

best_single = max(models, key=lambda k: auc(ytr, oof[k]))
print(f"best single model ({best_single})   OOF AUC {auc(ytr, oof[best_single]):.4f} | test AUC {auc(yte, test[best_single]):.4f}")
for k, (o, t) in blends.items(): print(f"{k:26s} OOF AUC {auc(ytr, o):.4f} | test AUC {auc(yte, t):.4f}")
print("hill-climbing weights:", {k: round(float(v), 2) for k, v in zip(models, w_hc)}); print("optimized weights    :", {k: round(float(v), 2) for k, v in zip(models, w_opt)})
assert auc(yte, blends["5. stacking (logistic)"][1]) > auc(yte, blends["1. average"][1])

Read the OOF and test columns together.

- The equal-weight blends (1 and 2) can be **worse** than the best single model: one weak member drags them down. Equal weights assume the models are about equally good.
- The fitted blends (3, 4, 5) give the weak model a weight near zero and land close to, or slightly above, the best single model. How much a blend gains depends on having several strong models that make different mistakes; here one model is clearly ahead, so the gain is small.
- Methods 3 and 4 fit their weights on the same OOF predictions they are scored on, so their OOF number is a little optimistic. The stacking row is honest because of the second round of folds.

### 3. How optimistic is a fitted blend?
Measure it: fit the weights on part of the OOF rows and score on the rest.

In [ ]:
gaps = []
for tr, va in StratifiedKFold(5, shuffle=True, random_state=2).split(oof, ytr):
    w = hill_climb(oof.values[tr], ytr[tr]); gaps.append(auc(ytr[tr], oof.values[tr] @ w) - auc(ytr[va], oof.values[va] @ w))
print(f"hill climbing: AUC on the rows used to fit the weights minus AUC on held-out rows = {np.mean(gaps):+.4f} ± {np.std(gaps):.4f}")

With five models the optimism is small. With five hundred candidate models, as in large teams' final days, it is not, and nested validation becomes essential.

### 4. What does each model add?
Remove one model at a time from the simple average.

In [ ]:
full = auc(ytr, oof.mean(1))
for name in models: print(f"without {name:12s} OOF AUC {auc(ytr, oof.drop(columns=name).mean(1)):.4f}   change {auc(ytr, oof.drop(columns=name).mean(1)) - full:+.4f}")

A weak model that disagrees with the others can add more than a strong model that agrees with them. Keep a model if removing it hurts, not because its own score is high.

### 5. Choosing two final submissions
Most competitions let you pick two. A sound pair:
1. your best cross-validation blend (trust CV),
2. something different and more conservative (a simpler blend, or the best model that does not depend on your riskiest idea).

Simulate the alternative: pick the two with the best *public* score.

In [ ]:
rng = np.random.default_rng(0)
candidates = {**{k: test[k].values for k in models}, **{k: t for k, (o, t) in blends.items()}}
cv_score = {**{k: auc(ytr, oof[k]) for k in models}, **{k: auc(ytr, o) for k, (o, t) in blends.items()}}
by_cv, by_public = [], []
for _ in range(300):
    pub = rng.random(len(yte)) < 0.2
    public = {k: auc(yte[pub], v[pub]) for k, v in candidates.items()}; private = {k: auc(yte[~pub], v[~pub]) for k, v in candidates.items()}
    pick_pub = sorted(public, key=public.get)[-2:]; pick_cv = sorted(cv_score, key=cv_score.get)[-2:]
    best_private = max(private.values())
    by_public.append(best_private - max(private[k] for k in pick_pub)); by_cv.append(best_private - max(private[k] for k in pick_cv))
print(f"regret against the best candidate on the private set: choose by public score {np.mean(by_public):.4f} | choose the two best by CV {np.mean(by_cv):.4f}")

Report the two numbers as they are. When candidates are this close, both rules lose little; the difference grows when the public set is small or when some candidates were tuned on it. The CV rule has one more advantage: it does not depend on luck you cannot measure.

### TODO
1. Blend your own OOF files from sessions 3 and 5 with all five methods.
2. For log loss instead of AUC, which blends still make sense? Try a geometric mean of probabilities.
3. Add ten copies of `hgb` with different seeds. What happens to the simple average, and to hill climbing?